# Lecture 3 – Data 100, Spring 2026

[Acknowledgments Page](https://ds100.org/sp26/acks/)

A demonstration of advanced `Polars` syntax to accompany Lecture 3.

In [1]:
import numpy as np
import polars as pl

## Dataset: California baby names

In today's lecture, we'll work with the `babynames` dataset, which contains information about the names of infants born in California.

The cell below pulls census data from a government website and then loads it into a usable form. The code shown here is outside of the scope of Data 100, but you're encouraged to dig into it if you are interested!

In [2]:
import urllib.request
import os.path
import zipfile

data_url = "https://www.ssa.gov/oact/babynames/state/namesbystate.zip"
local_filename = "data/babynamesbystate.zip"

# If the data exists don't download again
if not os.path.exists(local_filename): 
    with urllib.request.urlopen(data_url) as resp, open(local_filename, 'wb') as f:
        f.write(resp.read())

zf = zipfile.ZipFile(local_filename, 'r')

ca_name = 'STATE.CA.TXT'
field_names = ['State', 'Sex', 'Year', 'Name', 'Count']
with zf.open(ca_name) as fh:
    babynames = pl.read_csv(fh, has_header=False, new_columns=field_names)

babynames.head()

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1910,"""Mary""",295
"""CA""","""F""",1910,"""Helen""",239
"""CA""","""F""",1910,"""Dorothy""",220
"""CA""","""F""",1910,"""Margaret""",163
"""CA""","""F""",1910,"""Frances""",134


In [3]:
babynames.sample(n=5)

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""M""",1959,"""Marlo""",7
"""CA""","""M""",1921,"""Rex""",15
"""CA""","""F""",1979,"""Cindy""",362
"""CA""","""F""",1965,"""William""",9
"""CA""","""M""",2012,"""Braxton""",61


In [4]:
df = babynames

What happens when we want to use more complicated boolean masks?

In [5]:
# Note: The parentheses surrounding the code make it possible to 
# break the code into multiple lines for readability. But this is
# still a lot of code just to check for four names...

(
    babynames.filter((pl.col("Name")=="Bella") |
                     (pl.col("Name")=="Alex") |
                     (pl.col("Name")=="Narges") |
                     (pl.col("Name")=="Lisa"))
)

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1923,"""Bella""",5
"""CA""","""F""",1925,"""Bella""",8
"""CA""","""F""",1932,"""Lisa""",5
"""CA""","""F""",1936,"""Lisa""",8
"""CA""","""F""",1939,"""Lisa""",5
…,…,…,…,…
"""CA""","""M""",2018,"""Alex""",495
"""CA""","""M""",2019,"""Alex""",438
"""CA""","""M""",2020,"""Alex""",379


In [6]:
# A more concise method to achieve the above: The .is_in() method!
names = ["Bella", "Alex", "Narges", "Lisa"]
babynames.filter(pl.col("Name").is_in(names))

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1923,"""Bella""",5
"""CA""","""F""",1925,"""Bella""",8
"""CA""","""F""",1932,"""Lisa""",5
"""CA""","""F""",1936,"""Lisa""",8
"""CA""","""F""",1939,"""Lisa""",5
…,…,…,…,…
"""CA""","""M""",2018,"""Alex""",495
"""CA""","""M""",2019,"""Alex""",438
"""CA""","""M""",2020,"""Alex""",379


In [7]:
# What if we only want names that start with "N"?
babynames.filter(pl.col("Name").str.starts_with("N"))

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1910,"""Norma""",23
"""CA""","""F""",1910,"""Nellie""",20
"""CA""","""F""",1910,"""Nina""",11
"""CA""","""F""",1910,"""Nora""",6
"""CA""","""F""",1911,"""Nellie""",23
…,…,…,…,…
"""CA""","""M""",2022,"""Nilan""",5
"""CA""","""M""",2022,"""Niles""",5
"""CA""","""M""",2022,"""Nolen""",5


## Inspecting DataFrames: Useful Utility Functions

### Aggregation Methods

A `Series` carries the array functions you encountered in [Data 8](https://www.data8.org/su23/reference/#array-functions-and-methods) as methods you call on it.

In [8]:
yash_counts = babynames.filter(pl.col("Name")=="Yash")["Count"]
yash_counts

Count
i64
8
9
11
12
10
…
10
9
15


In [9]:
# Average number of babies named Yash each year
# Keep in mind that even if Python gives you 10 decimal places of precision, 
# you should think carefully about how much precision is meaningful!
# In this case, one decimal place or even no decimal places would be appropriate.
yash_counts.mean()

17.142857142857142

In [10]:
# Max number of babies named Yash born in any single year
max(yash_counts)

29

#### Built-In `Polars` Methods

There are many, *many* utility functions built into `Polars`, far more than we can possibly cover in lecture. You are encouraged to explore all the functionality outlined in the `Polars` [documentation](https://docs.pola.rs/api/python/stable/reference/index.html).

In [11]:
# A row is identified by its position in the table, counting from 0.
# with_row_index() writes those positions into a column of their own.
babynames.with_row_index().head()

index,State,Sex,Year,Name,Count
u32,str,str,i64,str,i64
0,"""CA""","""F""",1910,"""Mary""",295
1,"""CA""","""F""",1910,"""Helen""",239
2,"""CA""","""F""",1910,"""Dorothy""",220
3,"""CA""","""F""",1910,"""Margaret""",163
4,"""CA""","""F""",1910,"""Frances""",134


In [12]:
# Sorting hands out new positions, so record the current ones first if you want them later.
babynames.with_row_index("original_position").sort("Name").head()

original_position,State,Sex,Year,Name,Count
u32,str,str,i64,str,i64
366001,"""CA""","""M""",2008,"""Aadan""",7
369120,"""CA""","""M""",2009,"""Aadan""",6
384005,"""CA""","""M""",2014,"""Aadan""",5
398211,"""CA""","""M""",2019,"""Aadarsh""",6
362040,"""CA""","""M""",2007,"""Aaden""",20


In [13]:
# Returns the columns
babynames.columns

['State', 'Sex', 'Year', 'Name', 'Count']

In [14]:
# Returns the shape of the object in the format (num_rows, num_columns)
babynames.shape

(407428, 5)

In [15]:
# The total number of entries in the object, equal to num_rows * num_columns
babynames.height * babynames.width

2037140

In [16]:
# Returns the number of rows in the dataframe
len(babynames)

407428

In [17]:
# What summary statistics can we describe?
babynames.describe()

statistic,State,Sex,Year,Name,Count
str,str,str,f64,str,f64
"""count""","""407428""","""407428""",407428.0,"""407428""",407428.0
"""null_count""","""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,1985.733609,null,79.543456
"""std""",null,null,27.00766,null,293.698654
"""min""","""CA""","""F""",1910.0,"""Aadan""",5.0
"""25%""",null,null,1969.0,null,7.0
"""50%""",null,null,1992.0,null,13.0
"""75%""",null,null,2008.0,null,38.0
"""max""","""CA""","""M""",2022.0,"""Zyrus""",8260.0


In [18]:
# Our statistics are slightly different when working with categorical data (string columns)
babynames["Sex"].describe()

statistic,value
str,str
"""count""","""407428"""
"""null_count""","""0"""
"""min""","""F"""
"""max""","""M"""


In [19]:
# Randomly sample a row from the DataFrame.
babynames.sample()

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1917,"""Catherine""",135


In [20]:
# Rerun this cell a few times – you'll get different results!
babynames.sample(5)[:, 2:]

Year,Name,Count
i64,str,i64
2017,"""Hadlee""",6
2022,"""Donna""",21
1980,"""Sonia""",433
1963,"""Nelly""",5
1998,"""Natividad""",7


In [21]:
# Sampling with replacement requires the argument with_replacement=True.
# Sampling without replacement is the default behavior.
babynames[:10].sample(4, with_replacement=True)

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1910,"""Evelyn""",126
"""CA""","""F""",1910,"""Elizabeth""",93
"""CA""","""F""",1910,"""Helen""",239
"""CA""","""F""",1910,"""Alice""",118


In [22]:
# Multi-line chaining
result = (
    babynames.filter(pl.col("Year") == 2000)
    .sample(4, with_replacement=True)[:, 2:]
)
result

Year,Name,Count
i64,str,i64
2000,"""Kameron""",102
2000,"""Monika""",26
2000,"""Jorgeluis""",9
2000,"""Maribel""",129


In [23]:
# Count the number of times each unique value occurs in a Series,
# reporting the result as a two-column DataFrame ranked from most to least common.
babynames["Sex"].value_counts(sort=True)

Sex,count
str,u32
"""F""",239537
"""M""",167891


In [24]:
# Return all of the unique values in the Series.
babynames["Name"].unique()

Name
str
"""Kong"""
"""Keano"""
"""Tyren"""
"""Meghana"""
"""Tait"""
…
"""Attilio"""
"""Kamille"""
"""Paloma"""


## Adding, Removing, and Modifying Columns

To add a column, hand `with_columns()` a `Series` or an expression under the name you want. The method returns a new table, so assign the result back.

In [25]:
# Create a Series of the length of each name.
babyname_lengths = babynames["Name"].str.len_chars()

# Add that series as a column named "name_lengths"
babynames = babynames.with_columns(name_lengths = babyname_lengths)

babynames

State,Sex,Year,Name,Count,name_lengths
str,str,i64,str,i64,u32
"""CA""","""F""",1910,"""Mary""",295,4
"""CA""","""F""",1910,"""Helen""",239,5
"""CA""","""F""",1910,"""Dorothy""",220,7
"""CA""","""F""",1910,"""Margaret""",163,8
"""CA""","""F""",1910,"""Frances""",134,7
…,…,…,…,…,…
"""CA""","""M""",2022,"""Zayvier""",5,7
"""CA""","""M""",2022,"""Zia""",5,3
"""CA""","""M""",2022,"""Zora""",5,4


To modify a column, build the new values as an expression and pass it to `with_columns()` under the column's existing name.

In [26]:
# Modify the "name_lengths" column to be one less than its original value.
babynames = babynames.with_columns(name_lengths = pl.col("name_lengths") - 1)
babynames

State,Sex,Year,Name,Count,name_lengths
str,str,i64,str,i64,u32
"""CA""","""F""",1910,"""Mary""",295,3
"""CA""","""F""",1910,"""Helen""",239,4
"""CA""","""F""",1910,"""Dorothy""",220,6
"""CA""","""F""",1910,"""Margaret""",163,7
"""CA""","""F""",1910,"""Frances""",134,6
…,…,…,…,…,…
"""CA""","""M""",2022,"""Zayvier""",5,6
"""CA""","""M""",2022,"""Zia""",5,2
"""CA""","""M""",2022,"""Zora""",5,3


Rename a column using the `.rename()` method.

In [27]:
# Rename "name_lengths" to "Length".
babynames = babynames.rename({"name_lengths":"Length"})
babynames

State,Sex,Year,Name,Count,Length
str,str,i64,str,i64,u32
"""CA""","""F""",1910,"""Mary""",295,3
"""CA""","""F""",1910,"""Helen""",239,4
"""CA""","""F""",1910,"""Dorothy""",220,6
"""CA""","""F""",1910,"""Margaret""",163,7
"""CA""","""F""",1910,"""Frances""",134,6
…,…,…,…,…,…
"""CA""","""M""",2022,"""Zayvier""",5,6
"""CA""","""M""",2022,"""Zia""",5,2
"""CA""","""M""",2022,"""Zora""",5,3


Remove a column using `.drop()`.

In [28]:
# Remove our new "Length" column.
babynames = babynames.drop("Length")
babynames

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1910,"""Mary""",295
"""CA""","""F""",1910,"""Helen""",239
"""CA""","""F""",1910,"""Dorothy""",220
"""CA""","""F""",1910,"""Margaret""",163
"""CA""","""F""",1910,"""Frances""",134
…,…,…,…,…
"""CA""","""M""",2022,"""Zayvier""",5
"""CA""","""M""",2022,"""Zia""",5
"""CA""","""M""",2022,"""Zora""",5


## Sorting

In [29]:
# Sort a DataFrame – there are lots of Michaels in California.
babynames.sort("Count")

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1910,"""Adelaide""",5
"""CA""","""F""",1910,"""Adele""",5
"""CA""","""F""",1910,"""Adrienne""",5
"""CA""","""F""",1910,"""Althea""",5
"""CA""","""F""",1910,"""Antonia""",5
…,…,…,…,…
"""CA""","""M""",1970,"""Michael""",8196
"""CA""","""M""",1969,"""Michael""",8245
"""CA""","""M""",1990,"""Michael""",8246


In [30]:
# Sort a Series. Only the values come back, in their new order.
babynames["Name"].sort()

Name
str
"""Aadan"""
"""Aadan"""
"""Aadan"""
"""Aadarsh"""
"""Aaden"""
…
"""Zyrah"""
"""Zyrah"""
"""Zyrah"""


### Custom sorting

#### Approach 1: Create a temporary column

In [31]:
# Create a Series of the length of each name.
babyname_lengths = babynames["Name"].str.len_chars()

# Add a column named "name_lengths" that includes the length of each name.
babynames = babynames.with_columns(name_lengths = babyname_lengths)
babynames.head(5)

State,Sex,Year,Name,Count,name_lengths
str,str,i64,str,i64,u32
"""CA""","""F""",1910,"""Mary""",295,4
"""CA""","""F""",1910,"""Helen""",239,5
"""CA""","""F""",1910,"""Dorothy""",220,7
"""CA""","""F""",1910,"""Margaret""",163,8
"""CA""","""F""",1910,"""Frances""",134,7


In [32]:
# Sort by the temporary column.
babynames = babynames.sort(by="name_lengths", descending=True)
babynames.head(5)

State,Sex,Year,Name,Count,name_lengths
str,str,i64,str,i64,u32
"""CA""","""F""",1986,"""Mariadelosangel""",5,15
"""CA""","""M""",1987,"""Franciscojavier""",5,15
"""CA""","""M""",1988,"""Franciscojavier""",10,15
"""CA""","""M""",1989,"""Franciscojavier""",6,15
"""CA""","""M""",1991,"""Ryanchristopher""",7,15


In [33]:
# Drop the 'name_length' column.
babynames = babynames.drop("name_lengths")
babynames.head(5)

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1986,"""Mariadelosangel""",5
"""CA""","""M""",1987,"""Franciscojavier""",5
"""CA""","""M""",1988,"""Franciscojavier""",10
"""CA""","""M""",1989,"""Franciscojavier""",6
"""CA""","""M""",1991,"""Ryanchristopher""",7


#### Approach 2: Sorting on an expression

In [34]:
# Same as above, but the sort key is computed inline, so there is no temporary column to create and drop.
babynames.sort(pl.col("Name").str.len_chars(), descending=True).head()

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1986,"""Mariadelosangel""",5
"""CA""","""M""",1987,"""Franciscojavier""",5
"""CA""","""M""",1988,"""Franciscojavier""",10
"""CA""","""M""",1989,"""Franciscojavier""",6
"""CA""","""M""",1991,"""Ryanchristopher""",7


#### Approach 3: Sorting Using the `map_elements` Method

We can also use `map_elements` if we want to use an arbitrarily defined Python function. Suppose we want to sort by the number of occurrences of "dr" plus the number of occurences of "ea".

In [35]:
# First, define a function to count 
# the number of times "dr" or "ea" appear in each name.
def dr_ea_count(string):
    return string.count('dr') + string.count('ea')

# Then, use `map_elements` to apply `dr_ea_count` to each name in the "Name" column.
babynames = babynames.with_columns(
    dr_ea_count = pl.col("Name").map_elements(dr_ea_count, return_dtype=pl.Int64)
)

# Sort the DataFrame by the new "dr_ea_count" column
# so we can see our handiwork.
babynames = babynames.sort(by="dr_ea_count", descending=True)
babynames.head()

State,Sex,Year,Name,Count,dr_ea_count
str,str,i64,str,i64,i64
"""CA""","""F""",1986,"""Deandrea""",6,3
"""CA""","""F""",1988,"""Deandrea""",5,3
"""CA""","""F""",1990,"""Deandrea""",5,3
"""CA""","""F""",1994,"""Leandrea""",5,3
"""CA""","""M""",1985,"""Deandrea""",6,3


In [36]:
# Drop the `dr_ea_count` column.
babynames = babynames.drop("dr_ea_count")
babynames.head(5)

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1986,"""Deandrea""",6
"""CA""","""F""",1988,"""Deandrea""",5
"""CA""","""F""",1990,"""Deandrea""",5
"""CA""","""F""",1994,"""Leandrea""",5
"""CA""","""M""",1985,"""Deandrea""",6


### Example: Common names in certain years

In [37]:
names_06 = babynames.filter(pl.col('Year') == 2006)
boys_06 = names_06.filter(pl.col('Sex') == 'M')
girls_06 = names_06.filter(pl.col('Sex') == 'F')

In [38]:
boys_06.sort('Count').tail(8)

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""M""",2006,"""Joshua""",3031
"""CA""","""M""",2006,"""Jose""",3096
"""CA""","""M""",2006,"""Andrew""",3127
"""CA""","""M""",2006,"""David""",3158
"""CA""","""M""",2006,"""Jacob""",3199
"""CA""","""M""",2006,"""Angel""",3690
"""CA""","""M""",2006,"""Anthony""",3776
"""CA""","""M""",2006,"""Daniel""",3830


In [39]:
girls_06.sort('Count').tail(8)

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",2006,"""Emma""",1546
"""CA""","""F""",2006,"""Sophia""",1983
"""CA""","""F""",2006,"""Natalie""",2227
"""CA""","""F""",2006,"""Samantha""",2350
"""CA""","""F""",2006,"""Mia""",2453
"""CA""","""F""",2006,"""Ashley""",2606
"""CA""","""F""",2006,"""Isabella""",2801
"""CA""","""F""",2006,"""Emily""",3104


In [40]:
names_80s = babynames.filter(pl.col('Year').is_in(list(range(1980, 1990))))
boys_80s = names_80s.filter(pl.col('Sex') == 'M')
girls_80s = names_80s.filter(pl.col('Sex') == 'F')

In [41]:
display(girls_80s.sort('Count').tail(8))
display(boys_80s.sort('Count').tail(8))

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1980,"""Jennifer""",5612
"""CA""","""F""",1981,"""Jennifer""",5628
"""CA""","""F""",1982,"""Jennifer""",5813
"""CA""","""F""",1983,"""Jennifer""",5830
"""CA""","""F""",1986,"""Jessica""",5964
"""CA""","""F""",1988,"""Jessica""",6493
"""CA""","""F""",1989,"""Jessica""",6541
"""CA""","""F""",1987,"""Jessica""",6846


State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""M""",1982,"""Michael""",7162
"""CA""","""M""",1985,"""Michael""",7216
"""CA""","""M""",1986,"""Michael""",7263
"""CA""","""M""",1983,"""Michael""",7397
"""CA""","""M""",1987,"""Michael""",7452
"""CA""","""M""",1984,"""Michael""",7479
"""CA""","""M""",1988,"""Michael""",7761
"""CA""","""M""",1989,"""Michael""",7858


## Slido Exercises

### Which of the following statements returns a DataFrame of the first 3 baby names for boys in 2006?

In [42]:
mask1 = (babynames['Year'] == 2006) & (babynames['Sex'] == 'M')
mask2 = (babynames['Year'] == 2006) | (babynames['Sex'] == 'M')

In [43]:
babynames.filter(mask1).select(['Name', 'Sex', 'Count'])

Name,Sex,Count
str,str,i64
"""Leandro""","""M""",46
"""Andreas""","""M""",36
"""Deandre""","""M""",36
"""Leandre""","""M""",7
"""Adrean""","""M""",8
…,…,…
"""Ab""","""M""",7
"""Sy""","""M""",6
"""Aj""","""M""",5


In [44]:
babynames.filter(mask1)[:3, 'Name']

Name
str
"""Leandro"""
"""Andreas"""
"""Deandre"""


In [45]:
babynames.filter(mask2).select(['Name', 'Sex', 'Count'])[:3]

Name,Sex,Count
str,str,i64
"""Deandrea""","""M""",6
"""Alexandrea""","""F""",19
"""Joseandres""","""M""",5


In [46]:
babynames.filter(mask2).select(['Name', 'Sex', 'Count']).head(3)

Name,Sex,Count
str,str,i64
"""Deandrea""","""M""",6
"""Alexandrea""","""F""",19
"""Joseandres""","""M""",5


In [47]:
babynames[['Name', 'Sex', 'Count']].filter(mask1)[:3]

Name,Sex,Count
str,str,i64
"""Leandro""","""M""",46
"""Andreas""","""M""",36
"""Deandre""","""M""",36


#### What does this result mean?

In [48]:
babynames['Name'].value_counts(sort=True)

Name,count
str,u32
"""Jean""",223
"""Francis""",221
"""Guadalupe""",218
"""Jessie""",217
"""Marion""",214
…,…
"""Ku""",1
"""Lo""",1
"""Wa""",1


#### Which of the following returns `'b'`?

In [49]:
df = pl.DataFrame({
    "a": ['c', 'b', 'a'],
    "b": ['b', 'c', 'a'],
    "c": ['c', 'a', 'b']
})
df

a,b,c
str,str,str
"""c""","""b""","""c"""
"""b""","""c""","""a"""
"""a""","""a""","""b"""


In [50]:
df.drop('a')['b'][1]

'c'

In [51]:
df

a,b,c
str,str,str
"""c""","""b""","""c"""
"""b""","""c""","""a"""
"""a""","""a""","""b"""


In [52]:
df.filter(pl.col('a').is_in(['b','c']))[0,1]

'b'

In [53]:
df

a,b,c
str,str,str
"""c""","""b""","""c"""
"""b""","""c""","""a"""
"""a""","""a""","""b"""


In [54]:
df.rename({'b':'a', 'a':'b'})[0,'b']

'c'

#### Which of the following extracts the rows of `df` where the values in column `A` are at least as big as the smallest value in column `B`?

In [55]:
# Sample DataFrame for experimentation!
df = pl.DataFrame({
    "A": [2, 3, 4],
    "B": [3, 5, 5],
})
df

A,B
i64,i64
2,3
3,5
4,5


In [56]:
# df.filter(pl.col('A') >= pl.col('B').min())

In [57]:
# df.filter(pl.col('A') >= df['B'].sort(descending=True)[0])

In [58]:
# df.filter(pl.col('A') >= df['B'].value_counts(sort=True).tail(1))